# Sensor Analysis
This notebook cleans the sensor data and finds the best machine learning model to predict if the sensor is ON or OFF.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression      # draws a straight-line boundary between classes
from sklearn.neighbors import KNeighborsClassifier        # looks at the closest data points and votes
from sklearn.naive_bayes import GaussianNB                 # uses probability, assumes features don't affect each other
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier  # both build many decision trees and combine them
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
# load the raw sensor readings (16 features + the Class column: 1 = ON, 0 = OFF)
df = pd.read_csv("Data8076.csv")
df.head()

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10,Feature11,Feature12,Feature13,Feature14,Feature15,Feature16,Class
0,408.57,614.59,902.20,101.87,616.00,-408.96,-12.83,504.36,31.37,148.64,-183.49,175.54,-753.55,-755.17,195.86,932.07,1
1,-214.14,-436.19,645.15,321.57,325.52,644.01,-836.16,-61.13,330.57,-752.20,-532.58,-42.59,691.06,244.71,285.69,-816.03,0
2,-657.87,-375.79,-893.09,-63.96,658.41,-997.54,620.92,-346.88,352.48,-808.52,912.31,-316.31,330.67,-797.29,340.66,-900.27,1
3,-345.07,-537.54,-961.71,39.60,-469.30,12.81,-75.32,83.36,-93.44,-243.67,-344.19,828.84,-532.95,-443.01,-456.80,13.21,1
4,-453.61,-272.28,-854.55,497.01,289.24,-847.38,122.45,843.12,696.73,-676.89,639.48,-303.17,-456.28,704.84,-88.39,915.12,0


In [3]:
X = df.iloc[:, :-1]  # every column except the last = the features
y = df.iloc[:, -1]    # the last column = the answer we want to predict (Class)

# drop rows where every single feature reads 0 (likely a dead/faulty sensor reading)
mask = (X != 0).any(axis=1)
df_clean = df[mask]

print("Cleaned rows:", df_clean.shape[0])
print("Number of feature columns:", X.shape[1])

Cleaned rows: 11076
Number of feature columns: 16


In [4]:
X = df_clean.iloc[:, :-1]
y = df_clean.iloc[:, -1]

# 80% of rows to train the models, 20% held back to test how well they learned
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# 5 different ways of guessing ON/OFF from the 16 sensor features:
models = {
    # Logistic Regression: fits a straight line/boundary that best separates ON from OFF
    0: LogisticRegression(max_iter=500),

    # K-Nearest Neighbors: for a new reading, finds the 5 most similar past readings
    # and predicts whatever class most of them were
    1: KNeighborsClassifier(),

    # Naive Bayes: uses probability theory, assumes each feature is independent of the others
    2: GaussianNB(),

    # Random Forest: builds many decision trees on random slices of the data,
    # then takes a majority vote across all of them
    3: RandomForestClassifier(),

    # Gradient Boosting: builds decision trees one at a time, each new tree
    # trying to fix the mistakes of the last one
    4: GradientBoostingClassifier()
}

results = {}

# train each model, then score it on the unseen test data
for code, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[code] = acc

results

{0: 0.6453068592057761,
 1: 0.5767148014440433,
 2: 0.6453068592057761,
 3: 0.6268050541516246,
 4: 0.644404332129964}

In [6]:
# pick whichever model code scored the highest accuracy
best_model_code = max(results, key=results.get)
print("Best model code:", best_model_code)

Best model code: 0


In [7]:
best_model = models[best_model_code]
y_pred = best_model.predict(X_test)

# rows = actual class, columns = predicted class
# top-left = correctly said OFF, bottom-right = correctly said ON
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[   0  786]
 [   0 1430]]


### Confusion Matrix Notes
The best model (Logistic Regression) never predicts class 0 — it always guesses class 1.  
Its accuracy (~64.5%) is basically just the proportion of class 1 in the data, not real predictive skill.  
This suggests the features in this dataset carry little to no signal for telling the sensor ON/OFF state apart.

In [8]:
print(df_clean.shape[0])
print(X.shape[1])
print(best_model_code)


11076
16
0
